## Лабораторная работа 15.
Задание:
Сделать визуализацию работы Resnet50 (как на видео https://www.youtube.com/watch?v=fZvOy0VXWAI&t=85s)
1. Установить torch, torchvision, скачать Resnet (pretrained=True)
2. Написать функцию get_features_map() (установить hook), которая "возвращает" результаты работы слоя layer4 (находится перед слоем avgpool)
3. Достать из модели матрицу весов "W" последнего (полносвязного) слоя (fc).
4. Сложить карты признаков (2048 штук для ResNet50), с коэффициентами w_i, как показано в "статье" - https://alexisbcook.github.io/posts/global-average-pooling-layers-for-object-localization/

In [ ]:
python -c 'from keras.applications.resnet50 import ResNet50; ResNet50().summary()'

SyntaxError: invalid syntax (2337233249.py, line 1)

In [ ]:
# simple implementation of CAM in PyTorch for the networks such as ResNet, DenseNet, SqueezeNet, Inception
# last update by BZ, June 30, 2021 - ADAPTED FOR NEWER PYTORCH

import io
from PIL import Image
from torchvision import models, transforms
import torch
import numpy as np
import cv2
import json
import os

LABELS_file = "/home/julia/Desktop/code/PAC/dogs/n02088466_10083"
image_file = "/home/julia/Desktop/code/PAC/dogs/n02088466_10083.jpg"

if not os.path.exists(LABELS_file):
    print(f"Файл с метками не найден: {LABELS_file}")
    print("Загружаем стандартные метки ImageNet...")

    import urllib.request
    # короче на случай если файл с метками который я скеачала окажется побитым то пможно
    # скачать стандартные метки из imagenet, там должны быть сотоветсивя между индексами класса и названиями объектов

    url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
    urllib.request.urlretrieve(url, "imagenet_classes.json")
    LABELS_file = "imagenet_classes.json"

# networks such as googlenet, resnet, densenet already use global average pooling at the end, so CAM could be used directly.
model_id = 2
if model_id == 1:
    net = models.squeezenet1_1(pretrained=True)
    finalconv_name = "features"  # this is the last conv layer of the network
elif model_id == 2:
    net = models.resnet50(pretrained=True)
    finalconv_name = "layer4"
elif model_id == 3:
    net = models.densenet161(pretrained=True)
    finalconv_name = "features"

net.eval()
# это режим оценки модели, он отключает dropout, batch normalization
# то есть модель не дообучается на этой картинке а использует то что она уже знает чтобы понять что перед ней


# hook the feature extractor
features_blobs = []


def hook_feature(module, input, output):
    features_blobs.append(output.detach().cpu().numpy())
    # output - то что вышло из последнего слоя те 2048 картинок 7*7
    # ddetach - отключить граф вычислений  то есть не запоминать последовательность действий
    # чтобы использовать меньше памяти и предотвратить случсайнфе изменения
    # переложили на процессор потому что нампай не читает с видеокарты


# тут проходимся по всем слоям сети находим нужный и к нему приставляем хук
# теперь каждый раз когда данные пройдут через этот слой хук сработает
# Register hook properly for newer PyTorch
for name, module in net.named_modules():
    if name == finalconv_name:
        module.register_forward_hook(hook_feature)
        break


# Давайте добавим отладочный код, чтобы увидеть размерности
params = list(net.parameters())
print(f"Всего параметров: {len(params)}")

# Посмотрим на последние два параметра
weight_param = params[-2]
bias_param = params[-1]

print(f"Тип params[-2]: {type(weight_param)}")
print(f"Размерность weight_param: {weight_param.shape}")
print(f"Размерность bias_param: {bias_param.shape}")

# Для ResNet-50 вы увидите:
# Размерность weight_param: torch.Size([1000, 2048])
# Размерность bias_param: torch.Size([1000])

# get the softmax weight
params = list(net.parameters())  # net.parameters() - все обучаемые параметры сети
weight_softmax = np.squeeze(params[-2].detach().cpu().numpy())
# берем предпоследний параметр. последний те -1 это смещение
# а -2 это должны быть веса
# убираем с них лишние размерности
# вообще  weight_param: torch.Size([1000, 2048]) и лишний тут как бы и нет но код писали и для других моеделей вдруг там есть
# убирает все размерности где размер = 1

def returnCAM(feature_conv, weight_softmax, class_idx):
    # generate the class activation maps upsample to 256x256
    size_upsample = (256, 256)
    bz, nc, h, w = feature_conv.shape
    output_cam = []
    for idx in class_idx:
        cam = weight_softmax[idx].dot(feature_conv.reshape((nc, h * w)))
        cam = cam.reshape(h, w)
        cam = cam - np.min(cam)
        cam_img = cam / np.max(cam)
        cam_img = np.uint8(255 * cam_img)
        output_cam.append(cv2.resize(cam_img, size_upsample))
    return output_cam


normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
preprocess = transforms.Compose(
    [transforms.Resize((224, 224)), transforms.ToTensor(), normalize]
)

# load test image
img_pil = Image.open(image_file).convert("RGB")
img_tensor = preprocess(img_pil)
img_batch = img_tensor.unsqueeze(0)

with torch.no_grad():
    logit = net(img_batch)

# load the imagenet category list
with open(LABELS_file, "r") as f:
    if LABELS_file.endswith(".json"):
        classes_data = json.load(f)
        # Обрабатываем разные форматы JSON
        if isinstance(classes_data, dict):
            if all(k.isdigit() for k in classes_data.keys()):
                # Формат: {"0": ["n01440764", "tench"], ...}
                classes = [item[1] for item in classes_data.values()]
            else:
                # Формат: {"tench": "n01440764", ...}
                classes = list(classes_data.keys())
        elif isinstance(classes_data, list):
            classes = classes_data
    else:
        # Если это текстовый файл с одной меткой на строку
        classes = [line.strip() for line in f.readlines()]

h_x = torch.nn.functional.softmax(logit, dim=1)[0]
probs, idx = torch.sort(h_x, descending=True)
probs = probs.numpy()
idx = idx.numpy()

# output the prediction
print("\nTop 5 predictions:")
for i in range(0, 5):
    if idx[i] < len(classes):
        print("{:.3f} -> {}".format(probs[i], classes[idx[i]]))
    else:
        print("{:.3f} -> Class #{}".format(probs[i], idx[i]))

# generate class activation mapping for the top1 prediction
if features_blobs:
    CAMs = returnCAM(features_blobs[0], weight_softmax, [idx[0]])

    # render the CAM and output
    if idx[0] < len(classes):
        print("\nOutput CAM.jpg for the top1 prediction: %s" % classes[idx[0]])
    else:
        print("\nOutput CAM.jpg for the top1 prediction: Class #%d" % idx[0])

    # Load the original image
    img = cv2.imread(image_file)
    if img is not None:
        height, width, _ = img.shape
        heatmap = cv2.applyColorMap(
            cv2.resize(CAMs[0], (width, height)), cv2.COLORMAP_JET
        )
        result = heatmap * 0.3 + img * 0.5
        cv2.imwrite("CAM.jpg", result)
        print("CAM.jpg saved successfully!")
    else:
        print(f"Could not load image: {image_file}")
else:
    print("No features captured!")

/home/julia/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/julia/.local/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Всего параметров: 161
Тип params[-2]: <class 'torch.nn.parameter.Parameter'>
Размерность weight_param: torch.Size([1000, 2048])
Размерность bias_param: torch.Size([1000])

Top 5 predictions:
0.976 -> Class #163
0.012 -> Class #165
0.009 -> Class #161
0.001 -> Class #168
0.001 -> Class #164

Output CAM.jpg for the top1 prediction: Class #163
CAM.jpg saved successfully!
